# Rejected Loans Data Cleaning

This notebook processes the  dataset, which contains loan applications that were instantly declined by Lending Club. We will clean this data so it can be used for Reject Inference or Exploratory Data Analysis.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load the Data
The dataset is large (1.7GB), so we will load it directly from the  folder.

In [2]:
df = pd.read_csv('../data/rejected_2007_to_2018Q4.csv')
print(f'Loaded {len(df):,} rows and {df.shape[1]} columns.')
df.head()

Loaded 27,648,741 rows and 9 columns.


,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,10%,481xx,NM,4 years,0.0
1,1000.0,2007-05-26,Consolidating Debt,703.0,10%,010xx,MA,< 1 year,0.0
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,10%,212xx,MD,1 year,0.0
3,6000.0,2007-05-27,waksman,698.0,38.64%,017xx,MA,< 1 year,0.0
4,1500.0,2007-05-27,mdrigo,509.0,9.43%,209xx,MD,< 1 year,0.0


## 2. Standardize Column Names
Let's rename the columns to match the conventions used in the main accepted dataset.

In [3]:
df = df.rename(columns={
    'Amount Requested': 'loan_amnt',
    'Application Date': 'application_date',
    'Loan Title': 'purpose',
    'Risk_Score': 'fico',
    'Debt-To-Income Ratio': 'dti',
    'Zip Code': 'zip_code',
    'State': 'state',
    'Employment Length': 'emp_length',
    'Policy Code': 'policy_code'
})
df.columns

Index(['loan_amnt', 'application_date', 'purpose', 'fico', 'dti', 'zip_code',
       'state', 'emp_length', 'policy_code'],
      dtype='str')

## 3. Data Engineering & Cleaning
Several columns are formatted as strings and need to be converted to numeric values.

In [4]:
# 1. Parse DTI from string to float (e.g. "10%" -> 10.0)
df['dti'] = df['dti'].astype(str).str.replace('%', '')
df['dti'] = pd.to_numeric(df['dti'], errors='coerce')

# 2. Clean Employment Length
import re
def extract_years(text):
    if pd.isna(text):
        return np.nan
    num = re.findall(r'\d+', str(text))
    return int(num[0]) if num else 0

df['emp_length'] = df['emp_length'].apply(extract_years)

# 3. Convert Application Date to datetime
df['application_date'] = pd.to_datetime(df['application_date'], errors='coerce')

## 4. Handle Missing Values
The most critical column for risk analysis is  (Risk_Score). Let's check how many missing values we have.

In [5]:
missing_fico = df['fico'].isna().sum()
print(f'Missing FICO scores: {missing_fico:,} ({missing_fico/len(df)*100:.1f}%)')

# Fill missing FICO scores with the median
median_fico = df['fico'].median()
df['fico'] = df['fico'].fillna(median_fico)
print(f'Filled missing FICO scores with median: {median_fico}')

# Fill missing DTI with median
df['dti'] = df['dti'].fillna(df['dti'].median())

# Fill missing emp_length with median
df['emp_length'] = df['emp_length'].fillna(df['emp_length'].median())

Missing FICO scores: 18,497,630 (66.9%)
Filled missing FICO scores with median: 637.0


## 5. Export Cleaned Dataset
Exporting the cleaned dataset to CSV.

In [6]:
df.to_csv('../data/cleaned_rejected_loans.csv', index=False)
print('Exported successfully to ../data/cleaned_rejected_loans.csv')

Exported successfully to ../data/cleaned_rejected_loans.csv
